<a href="https://colab.research.google.com/github/lifan149/notes/blob/main/fastai/Practical-Deep-Learning-for-Coders/license_plate_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.4 MB/s eta 0:00:00
Mounted at /content/gdrive


In [2]:
import os
from pathlib import Path

# 定义解压目标目录
# 目标目录和zip文件在同一个文件夹，unzip会自动将内容放在该目录下
# 所以，这里可以直接使用zip文件所在的目录作为目标目录
destination_dir = '/content/gdrive/MyDrive/DataSet/'

# 定义你的ZIP文件在Google Drive上的路径
zip_file_path = destination_dir + 'CBLPRD-330k_v1.zip'

# # 解压后的文件存放在Google Drive上的路径
directory_path = destination_dir + "CBLPRD-330k"

# deleted_count = 0
# for filename in os.listdir(destination_dir):
#     if filename.lower().endswith('.jpg'): # 使用 .lower() 确保匹配 .JPG, .jpg 等
#         file_path = os.path.join(destination_dir, filename)
#         try:
#             os.remove(file_path)
#             # print(f"已删除：{file_path}")
#             deleted_count += 1
#         except OSError as e:
#             print(f"删除文件 '{file_path}' 失败：{e}")

# print(f"删除完成。共删除了 {deleted_count} 个 .jpg 文件。")


if not os.path.isfile(zip_file_path) and os.path.exists(directory_path):
    print(f"文件 '{zip_file_path}' 已解压。")
else:
    if not os.path.exists(destination_dir):
        print(f"目标目录 '{destination_dir}' 不存在，正在创建...")
        os.makedirs(destination_dir) # 创建目录，包括所有缺失的父目录
        print(f"目录 '{destination_dir}' 创建成功。")
    else:
        print(f"目标目录 '{destination_dir}' 已存在。")

    # 4. 执行解压操作
    print(f"正在将 '{zip_file_path}' 解压到 '{destination_dir}'...")
    # -q: 安静模式，不显示详细的解压过程
    # -d: 指定解压目录
    !unzip -q "$zip_file_path" -d "$destination_dir"

    print("解压完成！")

    # 5. 验证解压结果 (可选)
    # 列出目标目录下的内容，看看文件是否成功解压
    print(f"'{destination_dir}' 中的文件列表：")
    !ls -lh "$destination_dir"

print(f"'{destination_dir}' 中的文件列表：")
!ls -lh "$destination_dir"

count = 0
# os.walk 在 Python 3.5+ 内部已经使用 os.scandir() 进行优化
for root, dirs, files in os.walk(directory_path):
    for filename in files:
        # 避免额外的 is_file() 检查，因为 os.walk 已经区分了文件和目录
        if filename.lower().endswith('.jpg'):
            count += 1

print(f"在 '{directory_path}' 目录下 (包括子目录) 找到 {count} 个 .jpg/.JPG 文件 (最快递归方式)。")

目标目录 '/content/gdrive/MyDrive/DataSet/' 已存在。
正在将 '/content/gdrive/MyDrive/DataSet/CBLPRD-330k_v1.zip' 解压到 '/content/gdrive/MyDrive/DataSet/'...
解压完成！
'/content/gdrive/MyDrive/DataSet/' 中的文件列表：
total 1.4G
drwx------ 2 root root 4.0K May 30 05:26 CBLPRD-330k
-rw------- 1 root root 1.4G May 27 03:42 CBLPRD-330k_v1.zip
'/content/gdrive/MyDrive/DataSet/' 中的文件列表：
total 1.4G
drwx------ 2 root root 4.0K May 30 05:26 CBLPRD-330k
-rw------- 1 root root 1.4G May 27 03:42 CBLPRD-330k_v1.zip
在 '/content/gdrive/MyDrive/DataSet/CBLPRD-330k' 目录下 (包括子目录) 找到 329384 个 .jpg/.JPG 文件 (最快递归方式)。


In [3]:
from fastai.vision.all import *
from fastbook import *

dataset_root_path = Path(destination_dir)

### 定义字符集和映射

**为什么要定义字符集和映射**

在fastai教程识别熊（如灰熊、黑熊、泰迪熊）的例子中并没有定义字符集和映射，为什么车牌号识别需要？这是因为fastai 教程中训练识别熊的模型，使用的是 `CategoryBlock`，而车牌识别需要自定义字符映射，这是因为这两者的**任务类型和输出形式完全不同**。

### 熊分类模型 (Image Classification)

在识别熊（如灰熊、黑熊、泰迪熊）的例子中，这是一个典型的**图像分类**任务：

* **输入：** 一张图片。
* **输出：** 图片属于预定义类别中的**一个**（例如，`Grizzly` 或 `Black` 或 `Teddy`）。

回顾一下 Fastai 的 `DataBlock` 配置：

```python
bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(128)
)
```

这里发生了什么：

1.  **`ImageBlock`：** 明确告诉 Fastai，输入是图像。
2.  **`CategoryBlock`：** 明确告诉 Fastai，输出是一个**离散的类别标签**。
3.  **`get_y=parent_label`：** Fastai 在这里非常聪明地利用了文件系统结构。如果你的图片是这样组织的：
    ```
    images/
    ├── grizzly/
    │   ├── img1.jpg
    │   └── img2.jpg
    └── black/
        ├── img3.jpg
        └── img4.jpg
    ```
    那么 `parent_label` 函数会**自动提取父目录的名称作为类别标签**。例如，`images/grizzly/img1.jpg` 的标签就是 `'grizzly'`。

**Fastai 自动处理了什么？**

当 `CategoryBlock` 接收到像 `'grizzly'`、`'black'` 这样的字符串标签时，Fastai 在内部会**自动**为你完成字符到索引的映射：

* 它会收集所有唯一的类别字符串（例如，`['grizzly', 'black', 'teddy']`）。
* 然后它会为这些类别创建一个内部的“词汇表”（`vocab`），将每个类别字符串映射到一个唯一的整数索引（例如，`'grizzly' -> 0`, `'black' -> 1`, `'teddy' -> 2`）。
* 在训练时，你的标签会被转换为这些整数索引。
* 在预测时，模型输出的是这些整数索引，Fastai 会再次使用其内部的 `vocab` 将索引映射回类别字符串。

所以，你没有手动定义字符集和映射，是因为 Fastai 的 `CategoryBlock` 模块已经为你**封装了**这个“字符串类别到整数索引”的映射过程，并将其命名为 `vocab`。

### 车牌识别模型 (Image to Sequence / OCR)

而车牌识别则是一个**图像到文本序列**的任务：

* **输入：** 一张图片。
* **输出：** 一个**变长的字符序列**（例如，"京N12345"），而不是一个单一的离散类别。

你的车牌号 `云F5MTG学` 并不是一个单一的类别，它是多个字符组成的序列。车牌号的数量是无限的（字符的组合），你不可能为每一个可能的车牌号都定义一个类别。

因此，我们需要：

1.  **定义所有可能的单个字符**（`'京', '沪', ..., '0', '1', ..., 'A', 'B', ...`）。
2.  **将每个字符映射到唯一的整数ID。**
3.  **将整个车牌字符串（如“湘QSM5FJ”）编码成这些字符ID的序列。**
4.  模型会预测一个概率分布序列，每个时间步的概率分布对应你定义的字符集。
5.  **CTC Loss** 需要知道这些字符ID，尤其是空白字符的ID。
6.  **解码过程**需要将模型输出的字符ID序列转换回可读的车牌字符串。

Fastai 的 `CategoryBlock` 适用于单个分类标签，**不直接支持变长序列的字符识别**。虽然 Fastai 有 `TextBlock` 用于文本任务，但那是针对文本数据的（输入是文本，输出也是文本，或者文本分类），不直接适用于图像到文本的场景。

所以，在车牌识别中，我们必须**手动**定义字符集（所有可能的单个字符）以及它们到索引的映射，并创建自定义的 `PlateLabelBlock` 来处理这种序列标签。

### 总结核心区别：

* **熊分类：** 是**单标签分类**。Fastai 的 `CategoryBlock` 自动处理类别字符串到整数索引的映射（`vocab`）。
* **车牌识别：** 是**序列生成/识别**。输出是变长的字符序列，需要手动定义所有可能**单个字符**的集合以及它们到索引的映射，因为这是一个更底层的字符级预测任务。

In [4]:
# 定义所有可能的车牌字符
# 请注意，这里我将你提供的所有字符放到了一个列表中，确保没有重复
# 并且通常会添加一个 'blank' 字符用于 CTC Loss
CHARACTERS = [
    '京', '沪', '津', '渝', '冀', '晋', '蒙', '辽', '吉', '黑',
    '苏', '浙', '皖', '闽', '赣', '鲁', '豫', '鄂', '湘', '粤',
    '桂', '琼', '川', '贵', '云', '藏', '陕', '甘', '青', '宁',
    '新', '港', '澳', '挂', '学', '领', '使', '临',
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K',
    'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V',
    'W', 'X', 'Y', 'Z', 'I', 'O',
    '<blank>' # CTC Loss 需要一个空白字符
]

# enumerate()方法可以同时获取可迭代对象中元素的索引 (index) 和值 (value) 时
# 创建字符到索引的映射 (char_to_idx)
char_to_idx = {char: i for i, char in enumerate(CHARACTERS)}
print(f"char_to_idx的类型{type(char_to_idx)}值： {char_to_idx} ");

# 创建索引到字符的映射 (idx_to_char)
idx_to_char = {i: char for i, char in enumerate(CHARACTERS)}
print(f"idx_to_char的类型{type(idx_to_char)}值： {idx_to_char} ");


# 编码和解码函数
def encode_plate(plate_str):
    """将车牌字符串编码为索引序列."""
    return [char_to_idx[char] for char in plate_str]

def decode_plate(idx_seq):
    """将索引序列解码为车牌字符串."""
    # CTC 解码时可能会有重复字符和 blank，这里只是一个简单的解码
    # 实际 CTC 解码需要更复杂的逻辑，去除重复和空白字符
    return ''.join([idx_to_char[idx] for idx in idx_seq if idx != char_to_idx['<blank>']])

# 车牌类型（如果需要作为额外的分类任务，或者用于数据分析）
PLATE_TYPES = [
    '黑色车牌', '单层黄牌', '双层黄牌', '普通蓝牌', '拖拉机绿牌',
    '新能源大型车', '新能源小型车'
]
# 如果需要，也可以为车牌类型创建映射
plate_type_to_idx = {p_type: i for i, p_type in enumerate(PLATE_TYPES)}
print(f"plate_type_to_idx的类型{type(plate_type_to_idx)}值： {plate_type_to_idx} ");


char_to_idx的类型<class 'dict'>值： {'京': 0, '沪': 1, '津': 2, '渝': 3, '冀': 4, '晋': 5, '蒙': 6, '辽': 7, '吉': 8, '黑': 9, '苏': 10, '浙': 11, '皖': 12, '闽': 13, '赣': 14, '鲁': 15, '豫': 16, '鄂': 17, '湘': 18, '粤': 19, '桂': 20, '琼': 21, '川': 22, '贵': 23, '云': 24, '藏': 25, '陕': 26, '甘': 27, '青': 28, '宁': 29, '新': 30, '港': 31, '澳': 32, '挂': 33, '学': 34, '领': 35, '使': 36, '临': 37, '0': 38, '1': 39, '2': 40, '3': 41, '4': 42, '5': 43, '6': 44, '7': 45, '8': 46, '9': 47, 'A': 48, 'B': 49, 'C': 50, 'D': 51, 'E': 52, 'F': 53, 'G': 54, 'H': 55, 'J': 56, 'K': 57, 'L': 58, 'M': 59, 'N': 60, 'P': 61, 'Q': 62, 'R': 63, 'S': 64, 'T': 65, 'U': 66, 'V': 67, 'W': 68, 'X': 69, 'Y': 70, 'Z': 71, 'I': 72, 'O': 73, '<blank>': 74} 
idx_to_char的类型<class 'dict'>值： {0: '京', 1: '沪', 2: '津', 3: '渝', 4: '冀', 5: '晋', 6: '蒙', 7: '辽', 8: '吉', 9: '黑', 10: '苏', 11: '浙', 12: '皖', 13: '闽', 14: '赣', 15: '鲁', 16: '豫', 17: '鄂', 18: '湘', 19: '粤', 20: '桂', 21: '琼', 22: '川', 23: '贵', 24: '云', 25: '藏', 26: '陕', 27: '甘', 28: '青', 29: '宁', 30: 

### 解析数据集文件
加载 train.txt 和 val.txt 来获取图像路径和对应的车牌字符串

In [6]:
import pandas as pd

def load_data_from_txt(file_path, base_image_dir):
    """
    从数据文本文件加载图像路径和车牌标签。

    Args:
        file_path (Path): train.txt 或 val.txt 的路径。
        base_image_dir (Path): 图像文件所在的根目录（例如 CBLPRD-330k/）。

    Returns:
        pd.DataFrame: 包含 'image_path' 和 'plate_label' 的 DataFrame。
    """
    # 初始化一个空列表
    # 这个列表将用于临时存储从文本文件中解析出来的每一行数据，每行数据将是一个字典，包含图片完整路径和车牌号
    data = []
    # 以读模式 ('r') 打开指定的文件 (file_path)
    # 车牌字符包含中文，指定文件编码为 UTF-8，防止乱码
    with open(file_path, 'r', encoding='utf-8') as f:
      for line in f.readlines():
        line_split = line.strip().split(' ')
        if len(line_split) >= 2: # 确保至少有图像路径和车牌号
          img_full_path = base_image_dir / line_split[0]
          data.append({'image_path': img_full_path, 'plate_label': line_split[1]})
    return pd.DataFrame(data);



# 数据集文件的完整路径
# Python 标准库中的 pathlib 模块提供了一种面向对象的方式来处理文件系统路径
# pathlib.Path 对象与 / 运算符拼接时，无论路径末尾是否有 / (斜杠)，都完全没有关系**，都可以直接用 / 进行拼接
train_txt_path = dataset_root_path / 'train.txt'
val_txt_path = dataset_root_path / 'val.txt'

# 加载训练和验证数据
train_df = load_data_from_txt(train_txt_path, dataset_root_path);
val_df = load_data_from_txt(val_txt_path, dataset_root_path);

print(f"训练集大小: {len(train_df)}")
print(f"验证集大小: {len(val_df)}")
print("训练集前5行:\n", train_df.head())

print(os.path.exists(train_df.iloc[2].values[0]))



训练集大小: 325005
验证集大小: 17105
训练集前5行:
                                                   image_path plate_label
0  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/000272981.jpg    粤Z31632D
1  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/000204288.jpg    藏CFF7440
2  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/000390092.jpg     晋ZFSD44
3  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/000461632.jpg    苏GDG8575
4  /content/gdrive/MyDrive/DataSet/CBLPRD-330k/000278419.jpg    冀Q18266D
True


### 构建 Fastai DataBlock 和 DataLoader

#### **什么是 DataLoader**

在深度学习中，模型训练通常采用**小批量（mini-batch）**的方式进行。这意味着数据不是一次性全部喂给模型，而是分成一小批一小批地输入。`DataLoader` 就是 PyTorch提供的一种机制，用于：

* **批量处理 (Batching)：** 将单个数据样本（图像、文本、标签等）组合成固定大小的批次。
* **洗牌 (Shuffling)：** 在每个训练 epoch 开始时，随机打乱数据顺序，以防止模型学习到数据本身的顺序，提高泛化能力。
* **并行加载 (Parallel Loading)：** 通常使用多进程或多线程来在 CPU 上预加载数据批次，以便 GPU 在当前批次计算完成时能立即获得下一个批次，避免 I/O 阻塞造成的性能瓶颈。
* **数据转换 (Transforms)：** 在数据加载过程中应用必要的预处理和数据增强（例如图像缩放、裁剪、归一化、随机旋转等）。

#### **为什么需要 `DataLoader`？**

1.  **内存效率：** 大多数数据集无法一次性全部加载到内存中，特别是图像数据集。`DataLoader` 以批次的形式加载，大大降低了内存压力。
2.  **GPU 利用率：** GPU 运算速度很快。如果没有 `DataLoader` 的预加载和批处理，GPU 可能会花大量时间等待数据，导致计算资源浪费。`DataLoader` 确保 GPU 始终有数据可处理。
3.  **泛化能力：** `DataLoader` 的洗牌机制有助于模型看到不同的数据组合，减少过拟合，提高泛化能力。
4.  **标准化流程：** 它提供了一个统一的接口来处理各种类型的数据（图像、文本、表格），使得模型训练代码可以复用。

---

#### **什么是DataBlock**

如果说 `DataLoader` 是 PyTorch 提供的数据管道执行者，那么 `DataBlock` 就是 Fastai 提供的数据管道**蓝图（Blueprint）或配置器**。它是一种高级抽象，旨在简化你定义“如何从原始数据中获取输入 (x) 和标签 (y)”这个过程。

`DataBlock` 是 Fastai 中用于定义**数据处理流程**的核心组件。它以一种声明式的方式，让你告诉 Fastai 你的数据集长什么样，以及你希望如何对其进行处理。

你可以把 `DataBlock` 想象成一个工厂的配置清单，它详细说明了：

* **输入是什么？(What is the input?)** - 例如，图像 (`ImageBlock`)，文本 (`TextBlock`)，表格数据 (`TabularBlock`)。
* **输出是什么？(What is the output?)** - 例如，类别 (`CategoryBlock`)，回归值 (`RegressionBlock`)，或者像你车牌识别例子中的自定义序列 (`PlateLabelBlock`)。
* **如何获取所有数据项？(How to get all items?)** - 例如，从文件夹获取文件列表 (`get_image_files`)，或者从 Pandas DataFrame 获取行 (`lambda df: df.iterrows()`)。
* **如何从一个数据项中获取输入 (x)？(How to get x from an item?)** - 例如，从路径加载图像，或者从 DataFrame 的某一列获取数据。
* **如何从一个数据项中获取标签 (y)？(How to get y from an item?)** - 例如，从父文件夹名提取类别，或者从 DataFrame 的某一列获取车牌号字符串，再进行编码。
* **如何划分训练集和验证集？(How to split data?)** - 例如，随机分割 (`RandomSplitter`)，或者根据提供好的训练/验证文件 (`ColSplitter`)。
* **在每个数据项上应用什么转换？(What transforms to apply to each item?)** - `item_tfms`，例如图像大小调整。
* **在每个批次上应用什么转换？(What transforms to apply to each batch?)** - `batch_tfms`，例如数据增强（旋转、裁剪、亮度调整）和归一化。

#### **为什么需要构建 `DataBlock`？**

1.  **自动化复杂的数据管道：** 深度学习的数据预处理往往包含多个步骤：加载、转换、增强、批量处理等。`DataBlock` 将这些步骤整合在一个统一的、声明式的接口中，你只需告诉 Fastai **做什么**，而不用关心**如何做**。
2.  **易于修改和实验：** 更改数据加载或预处理方式变得非常简单，只需修改 `DataBlock` 的几行配置，而不需要重写大量的数据加载代码。
3.  **避免重复代码：** 许多数据处理模式是通用的，`DataBlock` 将它们封装起来，减少了重复编写加载、洗牌、批处理代码的工作。
4.  **与 Fastai 的 `Learner` 无缝集成：** `DataBlock` 的输出可以直接喂给 Fastai 的 `Learner` 对象，启动训练过程。
5.  **提高可读性：** 明确的数据定义方式使得代码更易于理解，其他人可以一眼看出你的模型正在处理什么类型的数据以及如何处理。
6.  **智能推断：** Fastai 在 `DataBlock` 内部进行了一些智能推断。例如，如果你使用 `CategoryBlock`，它会自动构建词汇表（vocab）来将字符串类别映射到整数ID。

#### **`DataBlock` 到 `DataLoader` 的流程**

当你定义好 `DataBlock` 后，你需要调用它的 `.dataloaders()` 方法来实际创建 `DataLoaders`：

```python
# 1. 定义 DataBlock (数据管道的蓝图)
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(128)
)

# 2. 从 DataBlock 创建 DataLoaders (实际的数据加载器)
# dblock 接收数据源（例如文件夹路径或 DataFrame），然后根据其蓝图创建训练和验证 DataLoader
dls = dblock.dataloaders(source_data_path, bs=64)
# 或者对于你的车牌识别例子：
dls = dblock.dataloaders(train_df, val_df, bs=64)
```

`dblock.dataloaders()` 方法会根据你 `DataBlock` 中定义的所有规则，生成两个 `DataLoader` 对象：一个用于训练集，一个用于验证集。这两个 `DataLoader` 会负责在训练过程中源源不断地向模型提供批次数据。

**总结：**

* `DataLoader` 是 PyTorch 的概念，负责数据的批量、洗牌和并行加载。
* `DataBlock` 是 Fastai 的高级抽象，它**定义了从原始数据到 `DataLoader` 的整个数据处理流程**。它让你可以声明式地配置数据源、标签获取、分割、转换和增强等步骤，最终产出训练和验证所需的 `DataLoader`。
* 你需要构建它们，因为它们是深度学习数据处理和训练过程中**高效、灵活、可维护**的基石。



In [16]:
# 定义输入数据的转换 (图像预处理)
# 图像大小可以根据模型输入要求调整
# 图像缩放，method='squish' 不保持宽高比，直接缩放
item_tfms = [Resize(128, method='squish')]

# aug_transforms是 Fastai 提供的一个非常方便的函数，它返回一个预定义的数据增强转换列表
# size=128: 指定了数据增强操作后，图片的目标尺寸仍然是 128x128 像素。这是为了与 item_tfms 中的 Resize(128) 保持一致，确保最终模型接收到的输入尺寸是固定的。
# max_rotate=10: 图片最大随机旋转角度为 ±10 度。这有助于模型学习识别在不同倾斜角度下的车牌。
# max_zoom=1.1: 图片最大随机缩放比例为 1.1 倍。这意味着图片可能会被放大到原图的 1.1 倍，然后随机裁剪，模拟车牌在不同距离下的外观。
# max_lighting=0.2: 图片亮度（brightness）和对比度（contrast）的最大随机变化幅度。0.2 表示亮度或对比度可以在 ±20% 的范围内随机调整。这有助于模型在不同光照条件下的鲁棒性。
# p_affine=0.7: 仿射变换 (affine transforms) 的应用概率为 0.7（70%）。max_rotate 和 max_zoom 都是仿射变换的一部分。这意味着在训练过程中，每次批次处理时，有 70% 的概率对图片应用旋转和缩放这些仿射变换。这有助于模型学习识别在不同视角和大小下的车牌
# Normalize: 这是 Fastai 中用于图片归一化的转换类。
# imagenet_stats 是 Fastai 提供的一个预定义常量，它是一个元组，包含在 ImageNet 数据集上计算出的平均值 (mean) 和标准差 (std)
batch_tfms = [*aug_transforms(size=128, max_rotate=10, max_zoom=1.1, max_lighting=0.2, p_affine=0.7),
              Normalize.from_stats(*imagenet_stats)]

# 自定义 DataBlock 的 get_x 和 get_y 函数
def get_x(row): return row['image_path'] # x是图像路径

def get_y(row): return encode_plate(row['plate_label']) # y是编码后的车牌序列

# 自定义一个 LabelBlock 来处理序列标签 (Fastai 需要知道如何处理这个类型)
# Fastai 的 TensorCategory 是处理单个分类标签的，这里我们需要处理 TensorBase 类型的序列
# 我们将标签编码为 PyTorch tensor
# PlateLabelBlock 继承自 Fastai 的 Transform 基类。
# 在 Fastai 中，任何想要作为数据处理管道中一个步骤的自定义操作（比如你的标签编码）都需要继承 Transform 类。
# 继承 Transform 意味着你的类能够与 Fastai 的数据处理系统（如 DataBlock）无缝集成，享受其提供的批处理、GPU 移动、并行处理等便利
# 这个 PlateLabelBlock 的作用是：
# 它告诉 Fastai 的 DataBlock：当处理标签（y）时，如果你从 get_y 函数那里得到一个整数列表，请把它转换成一个 PyTorch 的 64 位整数张量。
class PlateLabelBlock(Transform):

  # 这是 Transform 类中的一个特殊方法。
  # encodes: Fastai 的 Transform 类使用 encodes 和 decodes 方法来定义数据的正向（编码）和反向（解码）转换。
  # encodes 方法就是定义了如何将原始输入 o 转换为模型可以使用的格式。
  # self: 类的实例本身，Python 中方法的标准第一个参数。
  # o:list这是类型提示 (type hint)，表示 encodes 方法期望接收一个名为 o 的参数，并且这个参数的类型是 list。
  # 在这个具体的上下文中，o 将会是你的 get_y 函数返回的编码后的车牌序列。get_y 函数会将字符串车牌（如“云F5MTG学”）转换为一个整数列表（如 [2, 5, 12, 10, 7, 3, 4]，每个数字代表一个字符的 ID）。所以，这里的 o 就是这个整数列表。
  def encodes(self, o:list):
    # 这是 Fastai 提供的函数（实际上是 PyTorch 的 torch.tensor 的封装），它将 Python 列表 o 转换为一个 PyTorch 张量 (Tensor)。
    # 张量是 PyTorch 处理数据（包括输入和标签）的基本数据结构。
    # .long(): 这是 PyTorch 张量的一个方法，它将张量的数据类型转换为 torch.long (64位整数)。
    # 对于分类任务的标签（离散的类别 ID）或序列任务的字符 ID，通常需要使用整数类型。long() 类型在 PyTorch 中常用于索引或表示类别 ID。
    return tensor(o).long() # 将列表编码为 LongTensor

# 创建 DataBlock
# Fastai 的 DataBlock 需要知道你的输入和输出是什么类型的 'Block'
# 对于图像输入，是 ImageBlock
# 对于序列输出，我们使用自定义的 PlateLabelBlock
# get_items 接收一个 DataFrame，返回一个包含所有行的列表
# splitter 用于划分数据集，这里我们已经有了 train_df 和 val_df
# get_x 和 get_y 定义如何从每一行获取输入和标签
dblock = DataBlock(
    blocks=(ImageBlock, TransformBlock(type_tfms=PlateLabelBlock())),
    get_items=lambda df: list(df.iterrows()),       # 这是一个匿名函数（lambda 函数），它告诉 Fastai原始数据源是 Pandas DataFrame，并且它应该逐行地从 DataFrame 中获取每一个数据样本。每一行（以 Pandas Series 的形式）都将作为后续 get_x 和 get_y 函数的 row 参数
                                              # 当 Fastai 调用 dblock.dataloaders(train_df, val_df, ...) 时，train_df 或 val_df 会作为 df 参数传递给这个 lambda 函数
                                              # df.iterrows() 是 Pandas DataFrame 的一个方法，它会返回一个迭代器，每次迭代产生 DataFrame 中的一行数据（以 (index, row_series) 的形式）
    splitter=ColSplitter(),                   # ColSplitter(): 这是一个 Fastai 内置的分割器。它是按照表格里的某个标记分班，比如表格有一列is_valid，如果 is_valid 是 True，那这行数据就去验证集；如果 is_valid 是 False，那这行数据就去训练集。
                                              # Fastai其他常用分割器比如RandomSplitter()随机抽一部分数据到验证集（最常用）
                                              # 我们已经有了 train_df 和 val_df，ColSplitter()的作用就是占位，当你在 dblock.dataloaders(train_df, val_df, ...) 中直接传入了 train_df 和 val_df 时，Fastai 会直接使用这两个 DataFrame 作为训练集和验证集。它不会再去查找 is_valid 列进行二次分割。
    get_x=get_x,
    get_y=get_y,
    item_tfms=item_tfms,                      # 每个单独数据项的转换采用之前定义的item_tfms
    batch_tfms=batch_tfms                     # 整个批次的转换采用之前定义的batch_tfms
)

# 这行代码是 DataBlock 定义的最终步骤，它将你之前所有的配置（如何获取图片，如何获取标签，如何进行预处理和增强等）付诸实践，生成真正用于模型训练和验证的数据加载器。
# dls: 这是最终生成的数据加载器集合（Fastai 的 DataLoaders 对象），它包含了两个 DataLoader：一个用于训练集，一个用于验证集。你会在模型训练时把这个 dls 对象直接传给 Fastai 的 Learner。
# bs=64：批次大小 (Batch Size)，bs:这是 batch size 的缩写，表示每个训练或验证批次中包含的样本数量。
dls = dblock.dataloaders(train_df, val_df, bs=64)

# 检查一个批次的数据形状
xb, yb = dls.one_batch()
print(f"图像批次形状: {xb.shape}") # (bs, C, H, W)
print(f"标签批次形状: {yb.shape}") # (bs, max_seq_len)




AssertionError: ColSplitter only works when your items are a pandas DataFrame

### 构建模型 (CRNN 结构)

### CRNN 是什么？（通俗解释：图片里的“文字识别专家”）

CRNN 就是一个分工明确的团队：

1.  **CNN（卷积神经网络）部分**：像一个**“视觉分析师”**。它的任务是盯着图片，找到文字在哪里，并把文字的视觉特征提取出来。比如，它能分辨出“这块地方像个‘A’字”，“那块地方像个‘1’字”。它把这些特征整理成一系列的“视觉报告”。
2.  **RNN/LSTM（循环神经网络/长短期记忆网络）部分**：像一个**“上下文理解员”**。它拿着视觉分析师的“视觉报告”，一份一份地看。它知道文字是按照顺序排列的，所以它会结合前面看到的字符（报告），来判断现在这个位置最可能是哪个字符。比如，它可能知道“京”后面经常跟着字母，或者“1”后面经常跟着数字。
3.  **FC（全连接层）/输出层**：像一个**“翻译官”**。它拿到上下文理解员的判断结果，最终翻译成每个位置最可能是哪个字符的概率，然后我们就能找出得分最高的字符，把它们连起来就是最终的车牌号。

**所以，CRNN 这种结构非常适合车牌识别：**

* **CNN 看局部特征**：图片中的每个字符都是局部特征。
* **RNN 看序列关系**：车牌是一个字符序列，字符之间有顺序和组合规律。

---

### 对于新手，如何选择模型结构？

选择模型结构是深度学习中一个艺术与科学的结合，对于新手来说，有以下几个步骤和建议：

1.  **理解任务类型：**
    * **分类**：图片是哪一类（猫、狗、熊）？`vision_learner` 很适合。
    * **目标检测**：图片里有什么，在哪里？（识别汽车、行人，并框出位置）需要专门的模型如 YOLO, Faster R-CNN。
    * **分割**：图片里每个像素属于哪个物体？（精确画出猫的轮廓）需要 U-Net 等模型。
    * **序列识别**：图片里有文字，文字是什么？（车牌、验证码）需要 CRNN, Transformer 等模型。
    * **你的车牌识别属于“序列识别”**，所以 CRNN 是一个非常自然且强大的选择。

2.  **从成熟的架构开始 (基线模型)：**
    * **不要从零开始设计：** 除非你是研究人员，否则不要尝试从零开始设计一个全新的模型。像 CRNN 这种架构是经过多年研究和实践验证的。
    * **寻找通用模式：** 针对你的任务类型（例如序列识别），搜索“图像序列识别模型”、“OCR 深度学习模型”，你会发现 CRNN 是一个被广泛采用的基线。
    * **参考开源项目和论文：** 看看别人在解决类似问题时用了什么模型。GitHub 上有很多成熟的 OCR 项目，它们通常会分享模型架构。

3.  **理解模型各部分的职责：**
    * 对于 CRNN：
        * **CNN (骨干网络)**：负责从图像中提取**通用且有用的视觉特征**。这部分可以用现成的**预训练模型**（如 ResNet、EfficientNet、MobileNet 等）。预训练模型在 ImageNet 这种大型数据集上学习过识别各种图像特征，可以直接拿来用，这叫**迁移学习**，能大大加速训练并提升性能。
        * **RNN/LSTM/GRU**：负责处理 CNN 提取出的特征序列，捕捉**序列中的上下文信息**。选择 LSTM 是因为它能很好地处理长期依赖问题。
        * **输出层**：根据 RNN 的输出，预测每个时间步（即图像的每个宽度位置）上最可能的字符。

4.  **从小处着手，逐步复杂化：**
    * **CNN：** 可以从简单的几层卷积和池化开始，像你代码中那样。但对于实际项目，更推荐直接使用 Fastai 或 PyTorch 提供的**预训练骨干网络**，例如 `resnet34` 或 `xresnet50`。你只需要对它们的输出层做一些调整，以适应 RNN 的输入。
    * **RNN：** 可以从一层 LSTM 开始，然后尝试双向 LSTM，再增加层数 (`num_layers`)，并引入 `dropout`。
    * **参数调整：** `hidden_size`（隐藏层大小）和 `dropout`（丢弃率）是重要的超参数，需要通过实验来找到最佳值。

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# 这个 CRNN 类继承自 nn.Module，这是 PyTorch 中所有神经网络模块的基类
class CRNN(nn.Module):
  # num_chars 是字符词汇表的大小，比如之前创建的CHARACTERS的长度
  # hidden_size 隐藏状态的维度或大小。默认值为 256
  # ropout=0.3 这是 Dropout（丢弃层）的概率。默认值为 0.3。0.3 或 0.5 是 Dropout 常见的经验值
  def __init__(self, num_chars, hidden_size=256, dropout=0.3):
    super().__init__()
    # CNN特征提取器 (self.cnn)
    # # 它的任务是把输入图片（例如 3通道 128x128）转换成一串特征序列
    self.cnn = nn.Sequential(

        # 输入: (批量大小, 3, 128, 128)
        # 第一个卷积层
        # nn.Conv2d: 这是 PyTorch 中用于执行二维卷积操作的模块
        # 第一个参数3 n_channels - 输入通道数，表示输入图像的通道数。对于彩色图像（RGB），通常是 3 个通道。通俗理解：输入图片有红、绿、蓝三层信息
        # 第二个参数32 out_channels - 输出通道数，表示这个卷积层会生成多少个特征图（feature maps）。每个特征图对应一个卷积核所提取的特定特征。
        # kernel_size=3 (卷积核大小)
        # padding=1 (填充) 在进行卷积操作之前，在输入图像的边缘周围添加一圈零值。这有助于保留图像边缘的信息，并控制输出特征图的尺寸，通俗理解：为了不让图片边缘的信息在扫描过程中被忽略，我们给图片四周额外加一圈空白（零值），这样卷积核即使在边缘也能完整地扫描到。
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        # ReLU（Rectified Linear Unit）激活函数的 PyTorch 实现
        # 作用：将输入中的负值全部变为零，正值保持不变。f(x) = max(0, x)
        # 引入非线性：没有激活函数，无论堆叠多少层卷积，都只是在执行线性变换，无法学习复杂的模式（比如图像中弯曲的边缘、不规则的形状）。ReLU 引入了非线性，使得神经网络能够学习和表示更复杂的、非线性的关系。
        # 稀疏性：将负值变为零，有助于创建“稀疏”的激活，这在某些情况下可以提高计算效率并减少过拟合。
        # inplace=True: 这是一个优化选项。它表示 ReLU 操作会直接修改输入张量，而不需要额外分配内存来存储输出。这可以节省内存
        nn.ReLU(inplace=True),
        # 这是 PyTorch 中用于执行二维最大池化操作的模块。池化层通常跟在卷积层和激活函数之后。
        # 作用：降采样（Downsampling）：减小特征图的尺寸（高度和宽度）；
        #      特征压缩/不变性：在每个池化区域内，只保留最显著的特征（最大值），丢弃次要信息。这使得模型对输入图像中微小的平移、旋转具有一定的不变性。
        #      通俗理解的理解想象你得到了 32 份“特征报告”（来自卷积层）。现在，你想对每份报告进行“总结”。你把每份报告分成一个个小方块，在每个小方块里，你只找出最重要的那个点（最大的数值），然后把其他点都扔掉。这样，你的报告就变得更小、更精炼了。
        # (kernel_size)表示池化窗口的大小是 2x2。通俗理解：你总结报告时，每个小方块是 2x2 的区域。
        # (stride):表示池化窗口每次移动的步长是 2。当 stride 与 kernel_size 相同时（例如这里都是 2），池化窗口会不重叠地移动。
        # 效果：这会导致输出特征图的高度和宽度都变为输入的一半。
        # 如果输入是 (bs, C, H, W)，输出就是 (bs, C, H/2, W/2)。
        # 通俗理解：每个小方块总结完后，你跳过两个单位再进行下一个小方块的总结，这样就能把报告缩小一半。
        nn.MaxPool2d(2, 2), # 图像尺寸减半
        # 组合起来的意义（CNN 的一个基本块）
        # 把这三层组合起来，形成了一个 CNN 的基本处理单元：
        # Conv2d：从原始像素中提取低级到高级的特征（例如，边缘、纹理、形状）。
        # ReLU：引入非线性，让模型能够学习更复杂的、非线性的模式。
        # MaxPool2d：压缩数据量（减少计算量），使模型对图像中物体的微小位置变化具有鲁棒性，并进一步提取最显著的特征。
        # 在你的 CRNN 模型中，这些“基本块”被堆叠起来，CNN 部分就是通过这样一层层地提取和压缩特征，最终将原始图片转换为一个高度被压缩、宽度保持的“特征序列”，以供后面的 RNN 层处理。

        # 2. 第二个卷积层
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2, 2), # 图像尺寸再减半

        # 3. 第三个卷积层
        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d((2, 1), (2, 1)), # 对高度进行池化，保持宽度不变 (因为车牌是长条形)
                                      # (2,1) 核意味着在高度上做2x的池化，宽度上1x，也就是不池化。
                                      # 为什么要这样？车牌是长条形，我们希望在宽度方向保留更多信息，
                                      # 而把高度方向的信息压缩，最终让特征图的高度变成1，方便送入RNN。
                                      # 现在是 (批量大小, 128, 16, 32)

        # 4. 第四个卷积层
        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.MaxPool2d((2, 1), (2, 1)),

        # 5. 第五个卷积层 (带Batch Normalization)
        nn.Conv2d(256, 256, kernel_size=3, padding=1),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True),
        nn.MaxPool2d((2, 1), (2, 1)), # 再次对高度池化

        # 6. 第六个卷积层 (带Batch Normalization)
        nn.Conv2d(256, 512, kernel_size=3, padding=1),
        nn.BatchNorm2d(512),
        nn.ReLU(inplace=True),
        nn.MaxPool2d((2, 1), (2, 1)), # 再次对高度池化

        # 7. 最后一个卷积层
        # 这一步很关键，它的 stride=(2,1) 和 kernel_size=2 会让高度最终变成1。
        nn.Conv2d(512, 512, kernel_size=2, stride=(2,1), padding=0), # 最终特征图 H=1
        nn.ReLU(inplace=True),
        # 最终CNN输出形状：(批量大小, 512, 1, 32) - 也就是说，高度被压缩到了1！

    )

    # RNN (LSTM) 层
    # CNN 输出的特征图需要展平为序列，然后送入RNN
    # 假设最终CNN输出特征图的高度为 H_prime=1，宽度为 W_prime
    # 那么每个时间步的特征维度是 512 * H_prime
    self.rnn = nn.Sequential(
        nn.LSTM(512, hidden_size, bidirectional=True, num_layers=2, dropout=dropout),
        # 双向 LSTM
        # num_layers=2: 两层 LSTM
        # dropout: 避免过拟合
    )

    # 输出层 (全连接层)
    # 每个时间步输出 num_chars 个概率
    # bidirectional=True 使得 hidden_size * 2
    self.fc = nn.Linear(hidden_size * 2, num_chars)

  # 根据PyTorch 模型的约定此方法在进行模型推理 (Inference) 时或者进行模型训练 (Training) 时会被调用
  def forward(self, x):
    # x: (bs, C, H, W)
    # 通过 CNN 特征提取器
    cnn_features = self.cnn(x)
    # self.cnn 是你在 __init__ 方法中定义的卷积神经网络序列。
    # 这一步将输入的图像 x 经过一系列卷积、激活和池化操作，提取出高层次的视觉特征。
    # 假设你的 CNN 最终将高度压缩到 1，那么 cnn_features 的形状会是：
    # (bs, C_out, 1, W_out) e.g., (64, 512, 1, 32)
    # C_out 是 CNN 最后一层的输出通道数（在这里是 512），W_out 是最终的宽度。

    # 移除高度为1的维度 (bs, C_out, W_out)
    cnn_features = cnn_features.squeeze(2)
    # `squeeze(2)` 会移除张量中维度为 1 的那个维度。
    # 由于你的 CNN 设计目标是将图像高度压缩到 1，那么第 2 个维度（索引从 0 开始）就会是 1。
    # 移除后，形状变为 (bs, C_out, W_out) e.g., (64, 512, 32)。
    # 如果你的 CNN 实际输出高度不是 1（如我们之前讨论的 2），那么这里挤压后形状会是 (bs, C_out, W_out)
    # 但如果实际高度不是1，这个 squeeze(2) 会报错，或者不会改变形状，你需要检查确认。
    # 比如如果你的 CNN 输出了 (64, 512, 2, 32)，squeeze(2) 不会改变它。
    # 最安全的方式是确保 CNN 的输出高度就是 1。

    # 调整形状以适应 RNN 输入
    cnn_features = cnn_features.permute(2, 0, 1)
    # `permute` 用于交换张量的维度顺序。
    # RNN (LSTM) 通常期望的输入形状是 (sequence_length, batch_size, input_size)。
    # 原始形状：(bs, C_out, W_out) --> 假设是 (N, C, W)
    # 交换后形状：(W_out, bs, C_out) --> 假设是 (W, N, C)
    # 在车牌识别中，W_out（宽度）代表了序列的长度（车牌有多少个字符位置），
    # bs（batch_size）是批次大小，
    # C_out（512）是每个时间步的特征维度。
    # e.g., (32, 64, 512) - 序列长度，批次大小，特征维度

    # 通过 RNN 处理序列
    rnn_features, _ = self.rnn(cnn_features)
    # self.rnn 是你在 __init__ 中定义的 LSTM 序列。
    # 它接收调整后的 cnn_features 作为输入，并处理这个特征序列，学习字符之间的上下文依赖关系。
    # rnn_features 的形状是 (sequence_length, batch_size, hidden_size * 2)，
    # 因为你使用了双向 LSTM (bidirectional=True)，所以隐藏状态维度是 hidden_size 的两倍。
    # _ 是 LSTM 返回的隐藏状态和细胞状态，通常在每次调用时我们不需要它们，所以用 _ 接收。

    # 通过全连接层得到最终输出
    # rnn_features: (seq_len, bs, hidden_size * 2)
    output = self.fc(rnn_features)
    # self.fc 是你在 __init__ 中定义的线性层。
    # 它将 RNN 输出的每个时间步的特征 (hidden_size * 2) 映射到 num_chars 个类别分数。
    # output 的形状是 (sequence_length, batch_size, num_chars)。
    # 这是模型对每个时间步（即车牌图片每个宽度位置）预测出的每个字符的对数概率（logits）。
    # 这个格式正好符合 CTC Loss 的输入要求 (T, N, C)。

    return output

# 实例化模型
model = CRNN(num_chars=len(CHARACTERS))


### **nn.MaxPool2d的池化**

nn.MaxPool2d 的完整调用通常是这样的：
```
nn.MaxPool2d(kernel_size, stride=None, padding=0, dilation=1, return_indices=False, ceil_mode=False)
```
(2, 1) 重复出现，实际上是对应了 kernel_size 和 stride 这两个参

你提出了一个非常好的问题，`nn.MaxPool2d((2, 1), (2, 1))` 这种写法确实让初学者感到困惑，因为它看起来有重复的参数。这里涉及到 `nn.MaxPool2d` 的两个主要参数：`kernel_size` 和 `stride`。

---

### `nn.MaxPool2d` 的参数解释

`nn.MaxPool2d` 的完整调用通常是这样的：
`nn.MaxPool2d(kernel_size, stride=None, padding=0, dilation=1, return_indices=False, ceil_mode=False)`

你看到的 `(2, 1)` 重复出现，实际上是对应了 `kernel_size` 和 `stride` 这两个参数。

**`kernel_size`**: **池化窗口的大小**。
    * 如果你只传入一个数字，比如 `nn.MaxPool2d(2)`，它表示池化窗口是 `2x2`。
    * 如果你传入一个元组，比如 `nn.MaxPool2d((H_k, W_k))`，它表示池化窗口的高度是 `H_k`，宽度是 `W_k`。

**`stride`**: **池化窗口每次移动的步长**。
    * 默认情况下，如果 `stride` 没有显式指定，它会**默认等于 `kernel_size`**。这意味着池化窗口将不重叠地移动。
    * 如果你传入一个数字，比如 `stride=2`，它表示在高度和宽度方向上步长都是 2。
    * 如果你传入一个元组，比如 `stride=(H_s, W_s)`，它表示在高度方向步长是 `H_s`，宽度方向步长是 `W_s`。


`nn.MaxPool2d((2, 1), (2, 1))` 实际上等同于：
`nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1))`

* **`kernel_size=(2, 1)`**:
    * 表示池化窗口的高度是 2 像素，宽度是 1 像素。
    * **通俗理解：** 池化操作在垂直方向（高度）上每 2 个像素中取一个最大值，而在水平方向（宽度）上，它只看 1 个像素（即不跨越宽度像素），所以宽度方向上不进行缩减。

* **`stride=(2, 1)`**:
    * 表示池化窗口在高度方向上每次移动 2 像素，在宽度方向上每次移动 1 像素。
    * **通俗理解：** 由于步长与核大小相同，这意味着在高度方向上，池化窗口是**不重叠地**移动，每次移动都跳过一整个窗口的区域，从而将高度**减半**。而在宽度方向上，池化窗口是**紧密相连地**移动（步长为1），所以宽度**不会被缩减**。


`nn.MaxPool2d((2, 1), (2, 1))` 并不是参数重复的错误写法。它明确指定了：

* `kernel_size`（池化窗口）是 `2x1`。
* `stride`（移动步长）也是 `2x1`。

这种配置的目的是在**高度方向上进行降采样（尺寸减半）**，而在**宽度方向上保持分辨率不变**，这对于处理长条形图像（如车牌）的序列识别任务至关重要。

### **第 7 层通过步长 (stride) 和核大小 (kernel_size) 如何计算？**

**卷积层输出尺寸计算公式：**

对于一个输入尺寸为 $H_{in} \times W_{in}$ 的图像，经过一个卷积层后，输出尺寸 $H_{out} \times W_{out}$ 的计算公式如下：

$$H_{out} = \left\lfloor \frac{H_{in} - K_H + 2P_H}{S_H} \right\rfloor + 1$$
$$W_{out} = \left\lfloor \frac{W_{in} - K_W + 2P_W}{S_W} \right\rfloor + 1$$

其中：
* $H_{in}, W_{in}$：输入特征图的高度和宽度。
* $K_H, K_W$：卷积核的高度和宽度。
* $P_H, P_W$：高度和宽度方向上的填充（padding）值。
* $S_H, S_W$：高度和宽度方向上的步长（stride）值。
* $\lfloor \cdot \rfloor$：向下取整符号。

**第 7 层具体计算**：

第 7 层的定义是：

`nn.Conv2d(512, 512, kernel_size=2, stride=(2,1), padding=0)`

假设输入到这一层的特征图高度 $H_{in}$ 是 `2` （根据我们之前计算，第六层池化后 $4 \to 2$）。

* $H_{in} = 2$
* $K_H = 2$ (来自 `kernel_size=2`)
* $P_H = 0$ (来自 `padding=0`)
* $S_H = 2$ (来自 `stride=(2,1)` 的第一个值)

代入公式计算 $H_{out}$：
$$H_{out} = \left\lfloor \frac{2 - 2 + 2 \times 0}{2} \right\rfloor + 1$$$$H_{out} = \left\lfloor \frac{2}{2} \right\rfloor + 1$$$$H_{out} = \lfloor 1 \rfloor + 1$$
$$H_{out} = 1 + 1 = 2$$

**所以，计算结果是正确的：这一层会将高度从 2 变为 1。**


### 定义损失函数

Fastai 没有内置 CTC Loss，所以我们需要手动定义一个。

CTC Loss，全称 **Connectionist Temporal Classification Loss (连接时序分类损失)**，是深度学习中一种非常重要的损失函数，特别适用于处理**序列到序列（Sequence-to-Sequence）问题**，其中输入序列和输出序列的**长度不一致**，且**没有显式的对齐关系**。



In [ ]:
import torch

# 标准的 PyTorch 模块 (nn.Module)
class CTCLossFlat(nn.Module):
    "Wrapper for `CTCLoss`"
    def __init__(self, blank_idx=None, **kwargs):
        super().__init__()
        # 设置 CTC Loss 所需的空白符 (blank character) 的索引
        self.blank_idx = blank_idx if blank_idx is not None else char_to_idx['<blank>']
        self.ctc_loss = nn.CTCLoss(blank=self.blank_idx, reduction='mean', zero_infinity=True, **kwargs)

    def forward(self, inputs, targets):
        # inputs: (T, N, C) - T: input_sequence_length, N: batch_size, C: num_classes (num_chars)
        # targets: (N, L) - N: batch_size, L: target_sequence_length (padded)

        # 对于 CTC Loss，输入需要对数 softmax
        log_probs = F.log_softmax(inputs, dim=2) # 沿类别维度进行 log_softmax

        # 预测序列长度 (通常是 CNN 输出的宽度)
        input_lengths = torch.full(size=(log_probs.size(1),), fill_value=log_probs.size(0), dtype=torch.long)
        # target_lengths: 实际标签序列的长度
        # 我们的 yb 已经是 padded 的 tensor，需要计算真实长度
        target_lengths = (targets != 0).sum(dim=1) # 假设 padding value 是 0 (这是 Fastai 的默认填充)
        # 注意：如果你的 padding value 不是 0，需要修改这里

        loss = self.ctc_loss(log_probs, targets, input_lengths, target_lengths)
        return loss

# 实例化损失函数
ctc_loss_func = CTCLossFlat(blank_idx=char_to_idx['<blank>'])

### 训练模型

使用 Fastai 的 `Learner` 进行训练。

In [ ]:
# 定义一个评估指标 (可选，但推荐)
# 计算字符错误率 (CER) 或字错误率 (WER) 对 CTC 更合适
# 这里我们先用一个简单的准确率占位，实际需要自定义一个 CTC 解码后的准确率
# def plate_accuracy(inp, targ):
#     # inp: (T, N, C) model output
#     # targ: (N, L) true labels
#     # 需要 CTC 解码 inp，然后与 targ 比较
#     # 这需要复杂的实现，这里只是一个占位符
#     return accuracy(inp.mean(0), targ.mean(1).long()) # 示意性，不准确

# 训练 Learner
learn = Learner(dls, model, loss_func=ctc_loss_func, metrics=[accuracy]) # accuracy 这里只是占位，需自定义

# 寻找合适的学习率
learn.lr_find()

# 训练模型
# 可以使用 fit_one_cycle 或 fine_tune
# fit_one_cycle 是一种非常有效的训练策略
learn.fit_one_cycle(10, lr_max=3e-3) # 10个 epoch，最大学习率为 3e-3

# 保存模型
learn.export('license_plate_ocr_model.pkl')

### 模型预测与评估

In [ ]:
learn_inf = load_learner('license_plate_ocr_model.pkl')

# CTC 解码函数 (这是一个简化的贪婪解码，实际生产可能需要 Beam Search)
def ctc_greedy_decode(output_tensor, idx_to_char_map, blank_idx):
    """
    对 CTC 模型的输出进行贪婪解码。

    Args:
        output_tensor (torch.Tensor): 模型的输出 (seq_len, num_chars) 或 (num_chars,)。
        idx_to_char_map (dict): 索引到字符的映射。
        blank_idx (int): 空白字符的索引。

    Returns:
        str: 解码后的字符串。
    """
    # 找到每个时间步概率最高的字符索引
    pred_indices = torch.argmax(output_tensor, dim=-1).cpu().numpy()

    # 去除重复字符和 blank
    decoded_chars = []
    prev_char_idx = -1
    for idx in pred_indices:
        if idx != blank_idx and idx != prev_char_idx: # 去除重复和空白
            decoded_chars.append(idx_to_char_map[idx])
        prev_char_idx = idx
    return ''.join(decoded_chars)

# 预测单张图片
test_img_path = Path('/content/gdrive/MyDrive/DataSet/CBLPRD-330k_v1/000208356.jpg') # 替换为你的测试图片路径

# 加载图片并进行与训练时相同的预处理
img = PILImage.create(test_img_path)
# 应用 item_tfms (resize)
img_processed = item_tfms[0](img) # 假设 Resize 是第一个 transform

# 将图片转换为批次形式 (增加 batch 维度)
# Fastai 的 predict 方法会自动处理这些，但这里演示手动预测
img_tensor = TensorImage(image2tensor(img_processed).float())
# 应用 batch_tfms (normalize)
img_tensor = batch_tfms[1](img_tensor) # 假设 Normalize 是第二个 transform

# 增加 batch 维度 (1, C, H, W)
img_batch = img_tensor.unsqueeze(0)


learn_inf.model.eval() # 设置模型为评估模式
with torch.no_grad():
    model_output = learn_inf.model(img_batch) # (seq_len, 1, num_chars)

# 提取并解码输出
# model_output 是 (T, N, C)，我们需要针对单个样本 (N=1)
decoded_plate = ctc_greedy_decode(model_output[:, 0, :], idx_to_char, char_to_idx['<blank>'])

print(f"原始图片路径: {test_img_path}")
print(f"预测车牌: {decoded_plate}")

# (可选) 可视化图片
img.to_thumb(128).show()